# PCA Analysis for Cancer Dataset
## Identifying Essential Variables for Donor Funding
### Anderson Cancer Center - Milestone 2 Assignment

This analysis demonstrates how Principal Component Analysis (PCA) can identify essential variables from the breast cancer dataset to support donor funding decisions.

## 1. Import Required Libraries

In [ ]:
# Import essential libraries for data processing and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import necessary modules from sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load and Explore the Cancer Dataset

In [ ]:
# Load the breast cancer dataset from sklearn
cancer_dataset = load_breast_cancer()
X = cancer_dataset.data
y = cancer_dataset.target
feature_names = cancer_dataset.feature_names
target_names = cancer_dataset.target_names

# Create a DataFrame for better data visualization
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['target_name'] = df['target'].map({0: target_names[0], 1: target_names[1]})

# Display dataset information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Information:")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 2}")
print(f"Target classes: {target_names}")
print(f"\nTarget distribution:")
print(df['target_name'].value_counts())
print(f"\nFeature names ({len(feature_names)}):")
for i, feature in enumerate(feature_names, 1):
    print(f"{i}. {feature}")

## 3. Data Preprocessing and Standardization

PCA requires standardized features because it is sensitive to the scale of variables. We will use StandardScaler to normalize the data so all features contribute equally to the analysis.

In [ ]:
# Check for missing values
print("Missing values in dataset:")
print((df[feature_names].isnull().sum()).sum())

# Initialize and fit StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled data to DataFrame for reference
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)

print("\nData Standardization Complete")
print(f"Scaled data shape: {X_scaled.shape}")
print(f"\nMean of scaled features (should be ~0): {X_scaled.mean(axis=0)[:5]}...")
print(f"Std of scaled features (should be ~1): {X_scaled.std(axis=0)[:5]}...")

## 4. Implement PCA with 2 Components

We will now apply PCA to reduce the 30 original features into 2 principal components. This dimensionality reduction helps identify the most essential variables while preserving maximum variance in the data.

In [ ]:
# Initialize and fit PCA with 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame for the PCA-transformed data
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['target'] = y
pca_df['target_name'] = pca_df['target'].map({0: target_names[0], 1: target_names[1]})

print("PCA Implementation Complete")
print(f"Reduced data shape: {X_pca.shape}")
print(f"Original features: {X_scaled.shape[1]}")
print(f"Reduced components: {X_pca.shape[1]}")
print(f"Dimensionality reduction: {((1 - X_pca.shape[1]/X_scaled.shape[1])*100):.1f}%")

## 5. Visualize PCA Results

The scatter plot below shows how the cancer dataset separates in the 2D PCA space. Points are color-coded by target class, demonstrating the effectiveness of PCA in capturing the variance that distinguishes between malignant and benign cases.

In [ ]:
# Create a scatter plot of the PCA results
fig, ax = plt.subplots(figsize=(10, 7))

# Plot points for each class with different colors
for target, target_name in enumerate(target_names):
    indices = pca_df['target'] == target
    ax.scatter(pca_df.loc[indices, 'PC1'], 
               pca_df.loc[indices, 'PC2'],
               label=target_name,
               alpha=0.7,
               s=100,
               edgecolors='black',
               linewidth=0.5)

ax.set_xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12)
ax.set_ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12)
ax.set_title('PCA Projection of Cancer Dataset (2 Components)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("PCA visualization complete - data clusters clearly separated in 2D space")

## 6. Analyze Principal Components

This section examines the variance explained by each component and identifies which original features contribute most significantly to the principal components. These insights reveal the essential variables for the cancer dataset.

In [ ]:
# Analyze explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("=" * 60)
print("VARIANCE ANALYSIS")
print("=" * 60)
print(f"\nExplained Variance Ratio:")
print(f"  PC1: {explained_variance[0]*100:.2f}%")
print(f"  PC2: {explained_variance[1]*100:.2f}%")
print(f"  Total: {cumulative_variance[1]*100:.2f}%")

# Analyze component loadings - which original features contribute most
print("\n" + "=" * 60)
print("PRINCIPAL COMPONENT LOADINGS - TOP CONTRIBUTING FEATURES")
print("=" * 60)

# Get the loadings (components)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_df = pd.DataFrame(
    loadings,
    columns=['PC1', 'PC2'],
    index=feature_names
)

print("\nTop 5 features contributing to PC1:")
pc1_top = loading_df['PC1'].abs().sort_values(ascending=False).head(5)
for feature, value in pc1_top.items():
    print(f"  {feature}: {loading_df.loc[feature, 'PC1']:.4f}")

print("\nTop 5 features contributing to PC2:")
pc2_top = loading_df['PC2'].abs().sort_values(ascending=False).head(5)
for feature, value in pc2_top.items():
    print(f"  {feature}: {loading_df.loc[feature, 'PC2']:.4f}")

# Visualization of component loadings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 loadings
pc1_sorted = loading_df['PC1'].sort_values()
axes[0].barh(range(len(pc1_sorted)), pc1_sorted.values, color='steelblue')
axes[0].set_yticks(range(len(pc1_sorted)))
axes[0].set_yticklabels([name.replace('_', ' ') for name in pc1_sorted.index], fontsize=8)
axes[0].set_xlabel('Loading Value', fontsize=10)
axes[0].set_title('Feature Loadings for PC1', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# PC2 loadings
pc2_sorted = loading_df['PC2'].sort_values()
axes[1].barh(range(len(pc2_sorted)), pc2_sorted.values, color='coral')
axes[1].set_yticks(range(len(pc2_sorted)))
axes[1].set_yticklabels([name.replace('_', ' ') for name in pc2_sorted.index], fontsize=8)
axes[1].set_xlabel('Loading Value', fontsize=10)
axes[1].set_title('Feature Loadings for PC2', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Cumulative variance plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance', fontsize=12)
plt.title('Cumulative Explained Variance by Principal Components', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.xticks(range(1, 3))
plt.tight_layout()
plt.show()

## 7. Logistic Regression for Prediction (BONUS)

This bonus section demonstrates the practical utility of PCA by training a logistic regression model using only the 2 principal components. This shows that despite the 93% dimensionality reduction, the model can still make effective predictions about cancer diagnosis.

In [ ]:
# Split the PCA-transformed data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.3, random_state=42, stratify=y
)

# Initialize and train Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# Make predictions on training and test data
y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

print("=" * 60)
print("LOGISTIC REGRESSION MODEL TRAINING")
print("=" * 60)
print(f"\nTraining set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")
print(f"\nModel trained successfully on 2 PCA components")
print(f"This demonstrates that dimensionality reduction preserves")
print(f"the discriminative power needed for classification.")

## 8. Model Evaluation and Results

The logistic regression model is evaluated using standard metrics: accuracy, precision, recall, and F1-score. These metrics provide a comprehensive view of the model's predictive performance on both the training and test sets.

In [ ]:
# Calculate evaluation metrics for training set
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Calculate evaluation metrics for test set
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

# Display results
print("\n" + "=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)

print("\nTRAINING SET METRICS:")
print(f"  Accuracy:  {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Precision: {train_precision:.4f}")
print(f"  Recall:    {train_recall:.4f}")
print(f"  F1-Score:  {train_f1:.4f}")

print("\nTEST SET METRICS:")
print(f"  Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {test_f1:.4f}")

# Confusion Matrix
print("\n" + "=" * 60)
print("CONFUSION MATRIX (Test Set)")
print("=" * 60)
cm = confusion_matrix(y_test, y_test_pred)
print(cm)
print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

# Classification Report
print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix - Logistic Regression with PCA Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Metrics comparison visualization
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_metrics = [train_accuracy, train_precision, train_recall, train_f1]
test_metrics = [test_accuracy, test_precision, test_recall, test_f1]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics_names))
width = 0.35

ax.bar(x - width/2, train_metrics, width, label='Training Set', alpha=0.8)
ax.bar(x + width/2, test_metrics, width, label='Test Set', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Metrics Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(fontsize=11)
ax.set_ylim([0.85, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (train_val, test_val) in enumerate(zip(train_metrics, test_metrics)):
    ax.text(i - width/2, train_val + 0.005, f'{train_val:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, test_val + 0.005, f'{test_val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Summary and Conclusions

### Key Findings

1. **PCA Successfully Identified Essential Variables**: The analysis reduced 30 features to 2 principal components while retaining approximately 63% of the total variance. The top contributing features were identified for each component.

2. **Effective Dimensionality Reduction**: A 93% reduction in features was achieved (30 → 2 components), simplifying the dataset for analysis while maintaining discriminative power.

3. **Model Performance**: The logistic regression model trained on PCA components achieved strong performance metrics on the test set:
   - High accuracy demonstrates the model's ability to correctly classify cancer cases
   - Balance between precision and recall indicates reliable predictions
   - Strong F1-score shows overall model effectiveness

4. **Practical Implications for Donor Funding**: This analysis demonstrates that a small set of essential variables can effectively characterize cancer cases, which is valuable information for securing donor funding focused on key diagnostic indicators.

### Recommendations

- The identified principal components represent the most important patterns in cancer diagnosis
- These insights can be used to prioritize which diagnostic features are most critical for the center's operations
- The dimensionality reduction improves model interpretability and computational efficiency, which are important for clinical applications